In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import uuid

In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsproyecto")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/taxi/NYC.csv"

tabla_destino = f"{catalogo}.{esquema}.taxi_trips"

source_system = "KAGGLE_NYC_TAXI"

batch_id = str(uuid.uuid4())

print(f"Ruta origen    : {ruta}")
print(f"Tabla destino  : {tabla_destino}")
print(f"Source system  : {source_system}")
print(f"Batch ID       : {batch_id}")

Ruta origen    : abfss://raw@adlsproyecto.dfs.core.windows.net/taxi/NYC.csv
Tabla destino  : catalog_au.bronze.taxi_trips
Source system  : KAGGLE_NYC_TAXI
Batch ID       : 006fc23f-04f9-4ff3-bfea-d34937e02c79


In [0]:
df_taxi = spark.read.option('header', True)\
                    .option('inferSchema', True)\
                    .csv(ruta)

df_taxi.printSchema()

root
 |-- id: string (nullable = true)
 |-- vendor_id: integer (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- trip_duration: integer (nullable = true)



In [0]:
taxi_schema = StructType(fields=[
    StructField("id", StringType(), True),
    StructField("vendor_id", IntegerType(), True),
    StructField("pickup_datetime", TimestampType(), True),
    StructField("dropoff_datetime", TimestampType(), True),
    StructField("passenger_count", IntegerType(), True),
    StructField("pickup_longitude", DoubleType(), True),
    StructField("pickup_latitude", DoubleType(), True),
    StructField("dropoff_longitude", DoubleType(), True),
    StructField("dropoff_latitude", DoubleType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("trip_duration", LongType(), True)
])

In [0]:
df_taxi_final = spark.read\
    .option('header', True)\
    .option('timestampFormat', 'yyyy-MM-dd HH:mm:ss')\
    .schema(taxi_schema)\
    .csv(ruta)\
    .select(
        "*",
        col("_metadata.file_path").alias("_source_file")
    )

In [0]:
taxi_selected_df = df_taxi_final.select(
    col("id"),
    col("vendor_id"),
    col("pickup_datetime"),
    col("dropoff_datetime"),
    col("passenger_count"),
    col("pickup_longitude"),
    col("pickup_latitude"),
    col("dropoff_longitude"),
    col("dropoff_latitude"),
    col("store_and_fwd_flag"),
    col("trip_duration"),
    col("_source_file")
)

In [0]:
taxi_final_df = taxi_selected_df\
    .withColumn("_ingestion_timestamp", current_timestamp())\
    .withColumn("_source_system", lit(source_system))\
    .withColumn("_batch_id", lit(batch_id))


taxi_final_df = taxi_final_df.select(
    col("id"),
    col("vendor_id"),
    col("pickup_datetime"),
    col("dropoff_datetime"),
    col("passenger_count"),
    col("pickup_longitude"),
    col("pickup_latitude"),
    col("dropoff_longitude"),
    col("dropoff_latitude"),
    col("store_and_fwd_flag"),
    col("trip_duration"),
    col("_ingestion_timestamp"),
    col("_source_file"),
    col("_source_system"),
    col("_batch_id")
)

In [0]:
# No se requiere renombrado de columnas para taxi_trips,
# ya que los nombres del origen coinciden con el esquema Bronze.

In [0]:
taxi_final_df.write.mode("overwrite")\
    .insertInto(tabla_destino)